# 5교시 · 데이터 시각화
### — 남에게 보여 주기

앞 시간까지는 **내가 보려고** 숫자를 냈습니다.
이번 시간에는 **남이 보게** 만듭니다. 여기서부터는 성격이 달라집니다.

숫자는 틀리지 않았는데 그림 때문에 **상대가 잘못 이해하는 일**이 자주 생깁니다.
이번 시간에는 그림을 그리는 방법과 함께, **그림이 사람을 오해하게 만드는 방식**도 같이 봅니다.

**이 시간이 끝나면 할 수 있는 것**

1. 말하려는 내용에 맞는 차트 종류를 고를 수 있다
2. Matplotlib 으로 제목·축이름·범례를 직접 붙일 수 있다
3. Seaborn 으로 분포와 관계를 짧은 코드로 그릴 수 있다
4. Plotly 로 마우스를 올리면 값이 보이는 차트를 만들 수 있다
5. **같은 데이터로 정반대 인상을 주는 그래프를 만들 수 있고, 그래서 조심할 수 있다**

## 준비 — 데이터와 한글 폰트

이 노트북은 **혼자서도 처음부터 끝까지 실행되도록** 만들어져 있습니다.
4교시에서 이미 했더라도, 아래 세 셀을 **다시 한 번 실행**해 주세요.
Colab 은 노트북을 새로 열 때마다 폰트 설치가 초기화되기 때문입니다.

In [ ]:
import pandas as pd

BASE = 'https://raw.githubusercontent.com/JasonWhiteLee/ak-data-analysis-basics/main/'

orders = pd.read_csv(BASE + 'superstore_orders.csv', parse_dates=['Order Date', 'Ship Date'])
orders = orders.drop_duplicates()

print(orders.shape)

In [ ]:
!apt-get -qq install fonts-nanum > /dev/null

In [ ]:
import matplotlib.pyplot as plt
import matplotlib.font_manager as fm

fm.fontManager.addfont('/usr/share/fonts/truetype/nanum/NanumGothic.ttf')
plt.rc('font', family='NanumGothic')
plt.rc('axes', unicode_minus=False)      # 마이너스 기호도 깨지므로 함께 설정

plt.plot([1, 2, 3], [1, 4, 2])
plt.title('한글이 보이면 성공입니다')
plt.show()

---
# 5-1. 차트는 무엇을 위해 그리는가

먼저 표를 하나 만들어 보겠습니다. 지역별 매출입니다.

In [ ]:
지역별 = orders.groupby('Region')['Sales'].sum().sort_values(ascending=False)

지역별.round(0)

## 이 표를 그대로 보고해도 됩니다

숫자 네 개뿐입니다. 표로 봐도 어렵지 않습니다.
그런데 회의에서 이 표를 띄우면 사람들은 이렇게 합니다.

1. 801,749 를 읽는다
2. 739,516 을 읽는다
3. **머릿속에서 두 숫자를 비교한다**
4. 나머지 두 개도 같은 일을 반복한다

**표는 읽는 사람에게 계산을 시킵니다.** 그림은 그 계산을 대신 해 줍니다.

In [ ]:
지역별.plot(kind='bar', figsize=(7, 4))
plt.title('지역별 매출')
plt.show()

막대 길이를 보는 순간 순서와 차이가 **한 번에** 들어옵니다.
비교하는 일을 그림이 대신 해 줬기 때문입니다.

> ### 여기서 기억할 것
> 차트는 **예쁘게 만들려고** 그리는 것이 아닙니다.
> **읽는 사람이 머릿속에서 해야 할 비교를, 그림이 대신 해 주려고** 그립니다.
>
> 그래서 순서가 이렇게 됩니다.
> **① 내가 무엇을 말하려는지 정한다 → ② 그 말에 맞는 차트를 고른다 → ③ 그린다**
>
> 반대로 하면 안 됩니다. 일단 그려 놓고 "여기서 뭐가 보이지?" 하고 찾는 것은
> 나 혼자 탐색할 때는 괜찮지만, **남에게 보여 줄 차트를 만드는 방법은 아닙니다.**

| | 표 | 그림 |
|---|---|---|
| 정확한 값을 알려 준다 | ⭕ | ❌ |
| 크기 차이를 한눈에 보여 준다 | ❌ | ⭕ |
| 항목이 20개 넘어도 읽을 수 있다 | ⭕ | ❌ |
| 추세·패턴을 보여 준다 | ❌ | ⭕ |

**둘 중 하나가 더 좋은 게 아닙니다.** 정확한 숫자가 필요하면 표,
크기 비교나 흐름을 보여 주려면 그림입니다. 자료에 둘 다 넣어도 됩니다.

---
# 5-2. 무엇을 말하려는가에 따라 차트가 정해진다

차트 종류는 수십 가지가 있지만, **업무에서 쓰는 건 사실상 네 가지**입니다.
그리고 그 넷은 **말하려는 내용**으로 구분됩니다.

| 말하려는 것 | 예시 문장 | 차트 | 코드 |
|---|---|---|---|
| **비교** | "동부가 남부보다 두 배 판다" | 막대그래프 | `kind='bar'` |
| **추이** | "매출이 하반기에 올라간다" | 선그래프 | `kind='line'` |
| **관계** | "할인을 많이 하면 이익이 준다" | 산점도 | `kind='scatter'` |
| **분포** | "주문 대부분은 소액이다" | 히스토그램 | `kind='hist'` |

네 가지를 실제로 하나씩 그려 보겠습니다.

## ① 비교 — 막대그래프

**항목끼리 크기를 견주어 볼 때** 씁니다.
막대는 **길이**로 크기를 나타내기 때문에 사람 눈이 가장 정확하게 비교합니다.

In [ ]:
품목별 = orders.groupby('Sub-Category')['Sales'].sum().sort_values(ascending=False).head(10)

품목별.plot(kind='barh', figsize=(8, 5))
plt.title('매출 상위 10개 품목')
plt.xlabel('매출')
plt.show()

항목 이름이 길면 **`kind='barh'` (가로 막대)** 를 쓰세요.
세로 막대(`kind='bar'`)로 그리면 글자가 비스듬히 누워서 읽기 어렵습니다.

그리고 **정렬**을 했습니다. `sort_values(ascending=False)` 한 줄입니다.
정렬하지 않은 막대그래프는 순위를 읽으려면 눈이 왔다 갔다 해야 합니다.

## ② 추이 — 선그래프

**시간에 따라 어떻게 변했는지** 보여 줄 때 씁니다.
점을 선으로 이어 놓기 때문에 **"이어져 있다"** 는 느낌을 줍니다.
그래서 **가로축이 시간일 때만** 쓰는 것이 원칙입니다.

In [ ]:
orders['연월'] = orders['Order Date'].dt.to_period('M')
월매출 = orders.groupby('연월')['Sales'].sum()
월매출.index = 월매출.index.to_timestamp()      # 그림용으로 날짜형으로 되돌립니다

월매출.plot(kind='line', figsize=(10, 4))
plt.title('월별 매출')
plt.ylabel('매출')
plt.show()

위아래로 심하게 흔들립니다. 이 데이터는 **4년치(2023~2026)** 입니다.
전체적으로는 오른쪽으로 올라가지만, 달마다는 들쭉날쭉합니다.

**이 들쭉날쭉함을 어디까지 보여 줄지가 곧 5-7에서 다룰 문제입니다.**

## ③ 관계 — 산점도

**두 숫자가 서로 관련이 있는지** 볼 때 씁니다.
점 하나가 데이터 한 줄입니다. 여기서는 점 하나가 **주문 한 건**입니다.

In [ ]:
orders.plot(kind='scatter', x='Discount', y='Profit', figsize=(8, 5), alpha=0.3)
plt.title('할인율과 이익')
plt.axhline(0, color='red', linewidth=1)       # 이익 0 기준선
plt.show()

`alpha=0.3` 은 **점을 반투명하게** 만드는 설정입니다.
점이 1만 개나 겹치기 때문에, 이걸 안 하면 까맣게 뭉쳐서 아무것도 안 보입니다.

빨간 선(이익 0) **아래쪽에 있는 점이 적자 주문**입니다.
할인율이 오른쪽으로 갈수록 아래쪽 점이 많아지는 게 보입니다.

숫자로도 확인해 봅시다.

In [ ]:
print('할인율과 이익의 상관계수: {:.3f}'.format(orders['Discount'].corr(orders['Profit'])))
print('적자 주문 비율: {:.1f}%'.format((orders['Profit'] < 0).mean() * 100))

**상관계수 -0.219.** 마이너스니까 "할인이 커질수록 이익은 작아지는 쪽"입니다.
다만 -1 에서 한참 먼 값이라 **약한 관계**입니다. 할인만으로 이익이 정해지지는 않습니다.

> 상관계수는 -1 ~ +1 사이 값입니다. 0 에 가까우면 관계가 거의 없고,
> ±1 에 가까우면 한쪽이 변할 때 다른 쪽도 규칙적으로 변합니다.
> **자세한 내용은 6교시에서 다룹니다.**

## ④ 분포 — 히스토그램

**값들이 어디에 몰려 있는지** 볼 때 씁니다. 4교시에서 이미 그려 봤습니다.

In [ ]:
orders[orders['Sales'] < 1000]['Sales'].plot(kind='hist', bins=50, figsize=(9, 4))
plt.title('주문 금액 분포 (1,000 미만)')
plt.xlabel('주문 금액')
plt.show()

> ### 여기서 기억할 것
> **차트 종류를 고르는 일은 취향이 아닙니다.** 말하려는 내용이 정해 줍니다.
>
> - 항목 비교 → **막대**
> - 시간 흐름 → **선**
> - 두 숫자의 관계 → **산점도**
> - 값이 퍼진 모양 → **히스토그램**
>
> 하나만 더 — **원그래프(파이차트)는 되도록 쓰지 마세요.**
> 사람은 **각도**보다 **길이**를 훨씬 정확하게 비교합니다.
> 조각이 3개를 넘어가면 어느 쪽이 큰지 눈으로 판단이 안 됩니다. 그럴 땐 막대가 낫습니다.

---
# 5-3. Matplotlib 기본 — 직접 그리기

지금까지는 `df.plot(...)` 을 썼습니다. **판다스가 대신 그려 준 것**입니다.
편하지만, 세밀하게 손보려면 **Matplotlib 을 직접** 쓰는 편이 낫습니다.

`df.plot()` 도 사실은 내부에서 Matplotlib 을 부릅니다.
**같은 도구인데, 판다스를 거치느냐 직접 부르느냐의 차이**입니다.

In [ ]:
# 판다스에게 맡기기 — 짧지만 손볼 여지가 적습니다
지역별.plot(kind='bar')
plt.show()

제목도 없고, 축 이름도 없고, 크기도 작습니다.
**나 혼자 확인할 때는 이걸로 충분합니다.** 남에게 보여 줄 것은 아닙니다.

이제 직접 그려 보겠습니다.

In [ ]:
plt.figure(figsize=(8, 5))                        # ① 그림판 크기를 먼저 정합니다

plt.bar(지역별.index, 지역별.values, color='#4C72B0')   # ② 막대를 그립니다

plt.____('동부 지역이 전체 매출의 32.9%를 차지합니다')   # ③ 제목
plt.xlabel('지역')                                 # ④ 가로축 이름
plt.ylabel('매출')                                 # ⑤ 세로축 이름

plt.show()                                        # ⑥ 화면에 띄웁니다

## 한 줄씩 무슨 뜻인지

| 코드 | 하는 일 |
|---|---|
| `plt.figure(figsize=(8, 5))` | **그림판을 준비합니다.** 가로 8, 세로 5 (인치 단위) |
| `plt.bar(x, y)` | 막대를 그립니다. 선은 `plt.plot`, 점은 `plt.scatter` |
| `color='#4C72B0'` | 색을 지정합니다. `'red'` 처럼 이름으로도, `'#4C72B0'` 처럼 코드로도 됩니다 |
| `plt.title('...')` | 제목을 붙입니다 |
| `plt.xlabel` / `plt.ylabel` | 가로축·세로축 이름을 붙입니다 |
| `plt.show()` | **여기까지 그린 것을 화면에 띄웁니다.** 이걸 부르면 그림판이 비워집니다 |

**`plt.show()` 를 부르기 전까지 명령이 계속 같은 그림에 쌓입니다.**
그래서 `plt.bar` 로 막대를 그리고, 그 뒤에 `plt.title` 로 제목을 얹는 게 가능합니다.

`plt.show()` 를 안 쓰면 다음 셀의 그림과 겹쳐 그려질 수 있습니다. **항상 마지막에 넣으세요.**

## 범례 — 선이 여러 개일 때

한 그림에 선을 여러 개 그리면 **어느 선이 뭔지** 알려 줘야 합니다.
그게 **범례(legend)** 입니다.

`label=` 로 이름을 붙이고 `plt.legend()` 를 부르면 됩니다.

In [ ]:
카테고리별월매출 = orders.pivot_table(index='연월', columns='Category',
                                values='Sales', aggfunc='sum')
카테고리별월매출.index = 카테고리별월매출.index.to_timestamp()

plt.figure(figsize=(11, 4))

for 이름 in ['Furniture', 'Office Supplies', 'Technology']:
    plt.plot(카테고리별월매출.index, 카테고리별월매출[이름], label=이름)

plt.title('카테고리별 월 매출 추이')
plt.ylabel('매출')
plt.legend()                    # 이 한 줄이 범례를 만듭니다
plt.grid(alpha=0.3)             # 옅은 격자선 (값을 읽기 쉬워집니다)
plt.show()

선이 세 개나 얽혀 있어서 **읽기가 좋지는 않습니다.**
이럴 때는 선을 나누어 그리거나, **말하려는 한 개만 진하게** 하고 나머지는 회색으로 두는 방법이 있습니다.

아래가 그 방법입니다.

In [ ]:
plt.figure(figsize=(11, 4))

for 이름 in ['Furniture', 'Office Supplies']:
    plt.plot(카테고리별월매출.index, 카테고리별월매출[이름],
             color='lightgray', label=이름)

plt.plot(카테고리별월매출.index, 카테고리별월매출['Technology'],
         color='#C44E52', linewidth=2, label='Technology')

plt.title('Technology 매출의 진폭이 가장 큽니다')
plt.ylabel('매출')
plt.legend()
plt.show()

> ### 여기서 기억할 것
> **선을 다 똑같이 그리면 아무것도 강조되지 않습니다.**
> 말하려는 것 하나만 색을 주고 나머지를 회색으로 두면,
> 보는 사람의 눈이 **어디를 봐야 하는지** 알게 됩니다.
>
> 색은 **예쁘라고** 쓰는 게 아니라 **어디를 보라고** 쓰는 것입니다.

---
# 5-4. Seaborn — 짧은 코드로 분포와 관계

**Seaborn(시본)** 은 Matplotlib 위에 얹혀 있는 도구입니다.
Matplotlib 으로 열 줄 걸리는 걸 한 줄로 그려 줍니다. **Colab 에는 이미 설치돼 있습니다.**

Seaborn 함수들은 대부분 이 모양입니다.

```python
sns.무슨그림(data=표, x='가로축열', y='세로축열')
```

**표를 통째로 넘기고, 열 이름만 알려 주면 됩니다.**

In [ ]:
import seaborn as sns

sns.set_theme(style='whitegrid', font='NanumGothic')   # 보기 좋은 기본 설정 + 한글 폰트
plt.rc('axes', unicode_minus=False)

print(sns.__version__)

## ① 분포 — `sns.histplot`

In [ ]:
plt.figure(figsize=(9, 4))

sns.____(data=orders[orders['Sales'] < 1000], x='Sales', bins=50)

plt.title('주문의 절반이 53.7 이하입니다')
plt.xlabel('주문 금액')
plt.ylabel('주문 건수')
plt.show()

## ② 분포를 그룹별로 나눠 보기 — `hue`

Seaborn 의 진짜 장점은 **`hue=`** 입니다. **"이 열을 기준으로 색을 나눠 줘"** 라는 뜻입니다.
Matplotlib 으로 하려면 반복문을 써야 하는 일을, 인자 하나로 합니다.

In [ ]:
plt.figure(figsize=(9, 4))

sns.histplot(data=orders[orders['Sales'] < 500], x='Sales',
             hue='Segment', bins=40)

plt.title('고객 유형별 주문 금액 분포')
plt.xlabel('주문 금액')
plt.show()

## ③ 관계 — `sns.scatterplot`

In [ ]:
plt.figure(figsize=(9, 5))

sns.____(data=orders, x='Discount', y='Profit',
                hue='Category', alpha=0.4)

plt.title('할인율이 높아질수록 적자 주문이 늘어납니다')
plt.axhline(0, color='red', linewidth=1)
plt.show()

`hue='Category'` 하나로 **색 구분 + 범례**가 자동으로 붙었습니다.
Matplotlib 이었으면 카테고리마다 `plt.scatter` 를 부르고 `label=` 을 달아야 했습니다.

## ④ 비교 — `sns.barplot`

여기서 **주의할 점이 하나** 있습니다.
`sns.barplot` 은 기본적으로 **합계가 아니라 평균**을 그립니다. 게다가 막대 위에 **검은 세로선**이 붙습니다.

In [ ]:
plt.figure(figsize=(8, 4))

sns.barplot(data=orders, x='Region', y='Sales')

plt.title('지역별 평균 주문 금액 (합계가 아닙니다)')
plt.ylabel('평균 주문 금액')
plt.show()

막대 위의 검은 세로선은 **신뢰구간**입니다.
**"평균이 이 정도 범위 안에 있을 것 같다"** 는 뜻입니다. 6교시에서 다룹니다.

합계를 그리고 싶으면 **`estimator='sum'`** 을 주거나,
아예 `groupby` 로 미리 합쳐서 넘기면 됩니다.

In [ ]:
plt.figure(figsize=(8, 4))

sns.barplot(data=orders, x='Region', y='Sales',
            estimator='sum', errorbar=None,
            order=['East', 'West', 'Central', 'South'])   # 큰 순서로 직접 지정

plt.title('지역별 매출 합계')
plt.ylabel('매출 합계')
plt.show()

## ⑤ 상자그림 — `sns.boxplot`

4교시에 그렸던 상자그림입니다. Seaborn 으로 그리면 더 짧고 보기 좋습니다.

In [ ]:
plt.figure(figsize=(9, 5))

sns.boxplot(data=orders[orders['Sales'] < 1000], x='Category', y='Sales')

plt.title('카테고리별 주문 금액 (1,000 미만)')
plt.ylabel('주문 금액')
plt.show()

## Matplotlib 과 Seaborn, 언제 뭘 쓰나

| | Matplotlib | Seaborn |
|---|---|---|
| 코드 길이 | 길다 | 짧다 |
| 그룹별로 나눠 그리기 | 반복문 필요 | `hue=` 한 줄 |
| 세밀하게 손보기 | 자유롭다 | 제한이 있다 |
| 기본 모양 | 밋밋하다 | 보기 좋다 |

**둘 중 하나를 고르는 게 아닙니다.**
Seaborn 으로 그려 놓고 `plt.title` · `plt.ylabel` 로 다듬는 것이 실제로 가장 흔한 방식입니다.
위 코드들도 전부 그렇게 했습니다.

---
# 5-5. Plotly Express — 마우스를 올리면 값이 보이는 차트

지금까지 그린 것은 전부 **그림 파일**입니다. 움직이지 않습니다.
**Plotly(플롯리)** 로 그리면 **마우스를 올리면 값이 뜨고, 확대·축소가 되는 차트**가 나옵니다.

Colab 에 이미 설치돼 있고, 아래 코드를 실행하면 **노트북 안에서 바로 움직입니다.**

In [ ]:
import plotly.express as px

fig = px.bar(지역별.reset_index(), x='Region', y='Sales',
             title='지역별 매출 — 막대에 마우스를 올려 보세요')
fig.show()

막대에 마우스를 올리면 **정확한 숫자**가 뜹니다.
앞에서 "표는 정확한 값, 그림은 크기 비교"라고 했는데, **Plotly 는 그 둘을 동시에** 합니다.

In [ ]:
fig = ____(orders, x='Discount', y='Profit',
                 color='Category',
                 hover_data=['Sub-Category', 'Sales'],
                 opacity=0.5,
                 title='할인율과 이익 — 점 하나에 마우스를 올려 보세요')
fig.show()

`hover_data=['Sub-Category', 'Sales']` 를 넣었기 때문에,
**점 하나에 마우스를 올리면 그 주문의 품목과 매출까지** 보입니다.

4교시에서 "이상치 하나를 골라 판단하라"고 했었죠.
**이렇게 그려 놓으면 이상한 점을 클릭해서 바로 정체를 확인**할 수 있습니다.

한 개만 더 보겠습니다. 시간에 따라 움직이는 차트입니다.

In [ ]:
연도별지역 = orders.copy()
연도별지역['연도'] = 연도별지역['Order Date'].dt.year
연도별지역 = 연도별지역.groupby(['연도', 'Region'])['Sales'].sum().reset_index()

fig = px.line(연도별지역, x='연도', y='Sales', color='Region', markers=True,
              title='연도별 지역 매출 — 오른쪽 범례를 클릭해 보세요')
fig.show()

**오른쪽 범례에서 지역 이름을 클릭하면 그 선이 사라졌다 나타납니다.**
보고 싶은 것만 골라 볼 수 있습니다.

| | Matplotlib / Seaborn | Plotly |
|---|---|---|
| 결과물 | 그림 파일 (PNG) | 웹 화면 (움직임) |
| 보고서·PPT 에 붙이기 | 쉽다 | 캡처해야 한다 |
| 마우스로 값 확인 | 안 된다 | 된다 |
| 데이터 뜯어보기 | 불편하다 | 편하다 |

> ### 여기서 기억할 것
> **PPT·보고서에 넣을 것은 Matplotlib / Seaborn 으로 그리세요.**
> **혼자 데이터를 뒤져 볼 때는 Plotly 가 훨씬 빠릅니다.**
>
> 오늘은 "이런 게 있다"까지만 알면 충분합니다.

---
# 5-6. 제목을 결론 문장으로 쓰기

이 절은 코드가 거의 없습니다. 그런데 **효과는 오늘 배우는 것 중 가장 큽니다.**

대부분의 사람이 차트 제목을 이렇게 씁니다.

- ❌ 지역별 매출
- ❌ 월별 추이
- ❌ 할인율과 이익의 관계

이건 **제목이 아니라 이름표**입니다. 그림을 보면 이미 아는 내용입니다.
**아무 정보도 더해 주지 않습니다.**

제목 자리에는 **당신이 이 그림을 보고 내린 결론**을 쓰세요.

- ⭕ 동부 지역이 전체 매출의 32.9%를 차지합니다
- ⭕ 매출은 매년 오르지만 1~2월은 항상 저조합니다
- ⭕ 할인율 30%를 넘으면 대부분 적자입니다

**직접 비교해 봅시다.**

In [ ]:
# ❌ 이름표 제목
plt.figure(figsize=(8, 4))
plt.bar(지역별.index, 지역별.values, color='#8C8C8C')
plt.title('지역별 매출')
plt.show()

In [ ]:
# ⭕ 결론 제목 — 말하려는 것을 색으로도 함께 강조합니다
색 = ['#C44E52' if 지역 == 'East' else '#CCCCCC' for 지역 in 지역별.index]

plt.figure(figsize=(8, 4))
plt.bar(지역별.index, 지역별.values, color=색)
plt.title('동부 지역 한 곳이 전체 매출의 32.9%를 차지합니다')
plt.ylabel('매출')
plt.show()

## 두 그림의 데이터는 완전히 같습니다

바뀐 것은 **제목 한 줄과 색 하나**뿐입니다. 그런데 전달되는 내용이 다릅니다.

| | 이름표 제목 | 결론 제목 |
|---|---|---|
| 보는 사람이 하는 일 | 그림을 해석한다 | 결론을 확인한다 |
| 해석이 갈릴 가능성 | 있다 | 적다 |
| 만든 사람의 책임 | 애매하다 | **분명하다** |

마지막 줄이 중요합니다.
**결론을 제목에 쓰면, 그 결론이 틀렸을 때 내 책임이 됩니다.**

그래서 부담스럽습니다. "지역별 매출"이라고 써 두면 아무 말도 안 한 셈이니까요.
하지만 **아무 말도 안 하는 자료는 회의 시간만 씁니다.**

> ### 여기서 기억할 것
> **차트 제목은 결론 문장으로 씁니다.**
>
> 결론을 못 쓰겠다면, 그건 제목 문제가 아니라
> **아직 이 차트에서 무엇을 말할지 안 정해졌다는 뜻**입니다.
> 그럴 때는 제목을 고민하지 말고, **그 차트를 왜 그렸는지 다시 생각해 보세요.**

### 직접 해 보세요

아래 셀의 제목을 **결론 문장으로 바꿔** 실행해 보세요.
(먼저 그림을 보고, 무엇을 말할 수 있는지 정한 다음에 제목을 씁니다.)

In [ ]:
카테고리요약 = orders.groupby('Category').agg(
    매출=('Sales', 'sum'),
    이익=('Profit', 'sum')
)
카테고리요약['매출'] = 카테고리요약['매출'].round(0)
카테고리요약['이익'] = 카테고리요약['이익'].round(0)
카테고리요약['이익률'] = (카테고리요약['이익'] / 카테고리요약['매출'] * 100).round(1)

print(카테고리요약)

plt.figure(figsize=(8, 4))
plt.bar(카테고리요약.index, 카테고리요약['이익률'], color='#55A868')
plt.title('여기에 결론을 쓰세요')      # <-- 이 줄을 바꾸세요
plt.ylabel('이익률 (%)')
plt.show()

참고로, 위 표에서 읽을 수 있는 사실은 이렇습니다.

| Category | 매출 | 이익 | 이익률 |
|---|---|---|---|
| Furniture | 858,518 | 19,730 | **2.3%** |
| Office Supplies | 740,730 | 126,023 | 17.0% |
| Technology | 839,192 | 146,543 | 17.5% |

**Furniture 는 매출이 가장 많은데 이익률은 2.3% 밖에 안 됩니다.**
매출만 보고했다면 절대 안 보였을 사실입니다.

---
# 5-7. 오도하지 않을 책임

**이번 시간의 핵심입니다.**

지금부터 **같은 데이터로 정반대 인상을 주는 그래프를 두 쌍** 만들어 보겠습니다.
미리 말해 둡니다 — **어느 쪽도 데이터를 조작하지 않습니다.**
숫자를 고치지도, 빼지도, 지어내지도 않습니다. **전부 실제 값 그대로입니다.**

## 방법 ① — 세로축을 어디서 시작할 것인가

지역별 매출을 다시 봅니다. 동부 801,749, 서부 739,516.
**서부는 동부의 92.2%** 입니다. 차이는 8.4% 입니다.

먼저 **세로축을 0부터** 그립니다.

In [ ]:
plt.figure(figsize=(7, 4))
plt.bar(지역별.index, 지역별.values, color='#4C72B0')
plt.ylim(0, 900000)                        # 세로축을 0부터
plt.title('세로축 0부터 — 네 지역의 매출은 비슷한 수준입니다')
plt.ylabel('매출')
plt.show()

In [ ]:
plt.figure(figsize=(7, 4))
plt.bar(지역별.index, 지역별.values, color='#C44E52')
plt.____(380000, 820000)        # 세로축을 380,000부터 잘라냄
plt.title('세로축 잘라냄 — 동부가 남부의 두 배 넘게 팝니다')
plt.ylabel('매출')
plt.show()

## 무슨 일이 일어났습니까

| | 위 그림 (0부터) | 아래 그림 (380,000부터) |
|---|---|---|
| 동부 막대 | 길다 | 길다 |
| 남부 막대 | 조금 짧다 | **거의 없다** |
| 받는 인상 | 네 지역이 비슷하다 | **남부가 완전히 죽었다** |
| 실제 숫자 | 801,749 vs 391,507 | 801,749 vs 391,507 |

**숫자는 한 글자도 안 바뀌었습니다.** 바뀐 건 `plt.ylim` 한 줄입니다.

세로축을 잘라내면 **막대 길이의 비율이 실제 값의 비율과 달라집니다.**
남부 막대는 실제로 동부의 49% 인데, 아래 그림에서는 **길이가 거의 0** 으로 보입니다.

> **막대그래프의 세로축은 0에서 시작하는 것이 원칙입니다.**
> 막대는 **길이로 크기를 말하는** 그림이기 때문입니다. 길이를 자르면 거짓말이 됩니다.
>
> 선그래프는 사정이 다릅니다. 선은 크기가 아니라 **변화의 방향**을 말하기 때문에
> 축을 자르는 게 오히려 나을 때가 있습니다. 다만 그때도 **잘랐다고 밝혀야** 합니다.

## 방법 ② — 기간을 어디서 자를 것인가

이번엔 월별 매출입니다. 2026년 한 해만 봅니다. 실제 값은 이렇습니다.

In [ ]:
월매출_2026 = 월매출['2026']

월매출_2026.round(0)

12개월치가 전부 있습니다. 여기서 **일부만 잘라** 두 개의 그림을 만들겠습니다.

In [ ]:
성장 = 월매출['2026-07':'2026-11']

plt.figure(figsize=(8, 4))
plt.plot(성장.index, 성장.values, marker='o', color='#55A868', linewidth=2)
plt.title('7월 이후 매출이 157.5% 늘었습니다')
plt.ylabel('매출')
plt.show()

In [ ]:
하락 = 월매출['2026-04':'2026-12']

plt.figure(figsize=(8, 4))
plt.plot(하락.index, 하락.values, marker='o', color='#C44E52', linewidth=2)
plt.title('4월 이후 매출이 39.4% 줄었습니다')
plt.ylabel('매출')
plt.show()

## 두 제목 모두 **사실입니다**

| | 위 그림 | 아래 그림 |
|---|---|---|
| 기간 | 2026-07 ~ 2026-11 | 2026-04 ~ 2026-12 |
| 시작값 | 45,989 | 140,566 |
| 끝값 | 118,442 | 85,175 |
| 변화 | **+157.5%** | **-39.4%** |
| 제목 | "매출이 157.5% 늘었다" | "매출이 39.4% 줄었다" |

**두 문장 다 계산이 맞습니다.** 데이터도 같은 데이터입니다.
다른 건 **어디서 시작해서 어디서 끝냈는가** 하나뿐입니다.

이게 가능한 이유는, 이 데이터가 달마다 **심하게 오르내리기** 때문입니다.
그런 데이터에서는 **시작점만 잘 고르면 원하는 방향의 결론을 만들 수 있습니다.**

전체를 보면 이렇습니다.

In [ ]:
plt.figure(figsize=(10, 4))
plt.plot(월매출.index, 월매출.values, color='#4C72B0')
plt.title('전체 기간(2023~2026) — 오르내리지만 길게 보면 상승 추세입니다')
plt.ylabel('매출')
plt.show()

## 방법 ③ — 막대 순서를 어떻게 놓을 것인가

마지막입니다. 이건 좀 더 미묘합니다.

In [ ]:
품목5 = orders.groupby('Sub-Category')['Sales'].sum().nlargest(5)

plt.figure(figsize=(8, 4))
plt.bar(품목5.index, 품목5.values, color='#4C72B0')
plt.title('큰 순서로 정렬 — Chairs 가 1위라는 게 바로 보입니다')
plt.ylabel('매출')
plt.show()

In [ ]:
품목5_섞음 = 품목5.____        # 이름 가나다순으로 다시 늘어놓기

plt.figure(figsize=(8, 4))
plt.bar(품목5_섞음.index, 품목5_섞음.values, color='#8C8C8C')
plt.title('이름순 정렬 — 순위가 한눈에 안 들어옵니다')
plt.ylabel('매출')
plt.show()

아래 그림은 **거짓말을 하지 않습니다.** 그냥 **일을 안 한 것**입니다.

값이 들쭉날쭉 늘어서 있으면 보는 사람이 눈으로 순위를 매겨야 합니다.
**"읽는 사람의 비교를 대신해 준다"는 차트의 목적을 스스로 포기한 그림**입니다.

정렬은 사소해 보이지만, **하느냐 안 하느냐로 전달력이 갈립니다.**

## 정리 — 그래서 무엇을 조심해야 합니까

지금까지 만든 그림 중 **어느 것도 데이터를 조작하지 않았습니다.**
그런데 받는 인상은 정반대였습니다. **이게 시각화가 위험한 이유입니다.**

숫자를 고치면 그건 조작이고, 걸리면 큰일이 납니다. 그래서 아무도 안 합니다.
**그런데 축을 자르고 기간을 고르는 건 아무도 잘못이라고 안 합니다.** 효과는 거의 같은데요.

| 조심할 것 | 확인 질문 |
|---|---|
| 세로축 시작점 | 막대그래프인데 0에서 시작하지 않았습니까? |
| 기간 선택 | 왜 하필 그 달부터입니까? 다른 달부터 그리면 결론이 바뀝니까? |
| 정렬 순서 | 순위를 보여 주는 그림인데 정렬을 안 했습니까? |
| 제외한 데이터 | 이상치를 뺐다면 그 사실을 적었습니까? |
| 축 이름·단위 | 단위가 없거나, 두 그림의 축 범위가 다르지 않습니까? |

> ### 여기서 기억할 것
> **차트를 만든 사람은 보는 사람의 이해에 책임이 있습니다.**
>
> "나는 사실만 그렸다"는 변명이 안 되는 이유는,
> **어떤 사실을 어떻게 보여 줄지 고른 사람이 나이기 때문**입니다.
>
> 자기 차트를 남에게 보내기 전에 스스로 한 번 물어보세요.
> **"내가 이 그림을 처음 보는 사람이라면, 실제와 다르게 이해할 여지가 있는가?"**

---
# 5-8. 실습 — 차트 두 개와 그 한계 적기

## 실습 1. 차트를 그리고 결론 제목 달기

아래 빈 셀에 **차트를 두 개** 그리세요. 아래 목록에서 골라도 되고, 직접 정해도 됩니다.

| 소재 | 어울리는 차트 | 참고 코드 |
|---|---|---|
| 고객 유형(Segment)별 매출 | 막대 | `orders.groupby('Segment')['Sales'].sum()` |
| Sub-Category 별 이익률 | 가로 막대 | `groupby('Sub-Category').agg(...)` |
| 배송 방법(Ship Mode)별 배송 일수 분포 | 상자그림 | `sns.boxplot(...)` |
| 주문 수량(Quantity)과 이익의 관계 | 산점도 | `sns.scatterplot(...)` |
| 요일별 주문 건수 | 막대 | `orders['Order Date'].dt.dayofweek` |

**조건 두 가지입니다.**

1. **제목을 결론 문장으로** 쓸 것 (❌ "고객 유형별 매출" / ⭕ "Consumer 가 매출의 절반을 차지합니다")
2. **가로축·세로축 이름**을 붙일 것

In [ ]:
# 차트 ① — 여기에 코드를 쓰세요

In [ ]:
# 차트 ② — 여기에 코드를 쓰세요

### 여기에 적으세요

**이 텍스트 셀을 더블클릭**해서 채우세요.

1. 차트 ① 로 말하려는 **한 문장**은 무엇입니까


2. 차트 ② 로 말하려는 **한 문장**은 무엇입니까


3. 그 결론을 말하기 위해 **왜 그 차트 종류를 골랐습니까**


---

## 실습 2. 내 차트가 누군가를 오해하게 만들 수 있는 지점

방금 만든 두 차트 중 **하나를 고르세요.** 그리고 답하세요.

- **어느 차트입니까**


- **이 차트를 보고 사람들이 잘못 이해할 수 있는 지점은 어디입니까**
  (축 범위 / 기간 / 정렬 / 제외한 데이터 / 평균만 보여 준 것 등)


- **그 오해를 줄이려면 무엇을 바꾸거나 덧붙여야 합니까**


---

## 실습 3 (여유가 있으면). 오해하게 만드는 버전 직접 만들기

방금 고른 차트를 **일부러 과장된 버전으로 한 번 더** 그려 보세요.
`plt.ylim` 을 쓰든, 기간을 자르든 상관없습니다. **데이터는 절대 고치지 마세요.**

In [ ]:
# 과장된 버전 — 여기에 코드를 쓰세요

> ### 실습 3 을 시키는 이유
> **직접 만들어 봐야 남이 만든 걸 알아봅니다.**
>
> 앞으로 회의에서 남의 차트를 볼 때 세로축부터 보게 될 겁니다.
> 그게 이 실습의 목적입니다.

---
# 정리 — 오늘 쓴 것

## 코드

| 하는 일 | 코드 |
|---|---|
| 판다스로 빠르게 그리기 | `df['열'].plot(kind='bar')` |
| 그림판 크기 정하기 | `plt.figure(figsize=(8, 5))` |
| 막대 / 선 / 점 | `plt.bar(x, y)` · `plt.plot(x, y)` · `plt.scatter(x, y)` |
| 제목 · 축 이름 | `plt.title(...)` · `plt.xlabel(...)` · `plt.ylabel(...)` |
| 범례 | `plt.plot(..., label='이름')` + `plt.legend()` |
| 축 범위 | `plt.ylim(아래, 위)` |
| 기준선 | `plt.axhline(0)` · `plt.axvline(0)` |
| 화면에 띄우기 | `plt.show()` |
| Seaborn 기본 설정 | `sns.set_theme(style='whitegrid', font='NanumGothic')` |
| 분포 | `sns.histplot(data=df, x='열', hue='그룹')` |
| 관계 | `sns.scatterplot(data=df, x='열1', y='열2', hue='그룹')` |
| 비교 | `sns.barplot(data=df, x='그룹', y='값', estimator='sum')` |
| 상자그림 | `sns.boxplot(data=df, x='그룹', y='값')` |
| 동적 차트 | `import plotly.express as px` + `px.bar(...)` · `px.scatter(...)` |
| 한글 폰트 | `!apt-get -qq install fonts-nanum` + `plt.rc('font', family='NanumGothic')` |

## 차트 고르는 표

| 말하려는 것 | 차트 |
|---|---|
| 항목끼리 비교 | 막대 (항목 이름이 길면 가로 막대) |
| 시간에 따른 변화 | 선 |
| 두 숫자의 관계 | 산점도 |
| 값이 퍼진 모양 | 히스토그램 · 상자그림 |

## 남길 것 세 가지

1. **차트 종류는 말하려는 내용이 정한다** — 먼저 문장을 정하고, 그 다음에 차트를 고릅니다
2. **제목에는 이름표가 아니라 결론을 쓴다** — 결론을 못 쓰겠다면 그 차트를 왜 그렸는지 다시 생각합니다
3. **데이터를 조작하지 않아도 오해하게 만들 수 있다** — 축·기간·정렬만으로 인상이 뒤집힙니다

---

### 다음 시간

**6교시 · 그 차이, 진짜입니까?**

오늘 그린 그림에서 "동부가 더 많다", "할인이 많으면 이익이 준다" 같은 것을 눈으로 봤습니다.
다음 시간에는 이렇게 묻습니다.

**눈으로 본 그 차이가 우연히 생긴 것은 아닙니까?**

차이를 재는 방법과, 그 차이가 우연인지 판단하는 방법을 다룹니다.